In [7]:
import numpy as np
import pandas as pd

def create_block(block_name, n_trials, pc_ratio):
    """
    Generates a randomized block of Go/No-Go trials based on the target conflict ratio.
    """
    # 1.trial counts depends pc, changing pc to tweak probabilities
    n_pc = int(n_trials * pc_ratio)
    n_pi = n_trials - n_pc
    
    # Split PC equally into Go-to-Win (GW) and NoGo-to-Avoid-Loss (NAL)
    n_gw = n_pc // 2
    n_nal = n_pc - n_gw
    
    # Split PI equally into NoGo-to-Win (NW) and Go-to-Avoid-Loss (GAL)
    n_nw = n_pi // 2
    n_gal = n_pi - n_nw
    
    trials = []
    
    # 2. Build Pavlovian Congruent (PC) Trials
    # GW: Win Cue (+1), Optimal Action = Go (1)
    trials.extend([{'Block': block_name, 'Conflict': 'PC', 'Trial_Type': 'GW', 
                    'Cue_Valence': 1, 'Optimal_Action': 1}] * n_gw)
    # NAL: Avoid Cue (-1), Optimal Action = No-Go (0)
    trials.extend([{'Block': block_name, 'Conflict': 'PC', 'Trial_Type': 'NAL', 
                    'Cue_Valence': -1, 'Optimal_Action': 0}] * n_nal)
    
    # 3. Build Pavlovian Incongruent (PI) Trials
    # NW: Win Cue (+1), Optimal Action = No-Go (0)
    trials.extend([{'Block': block_name, 'Conflict': 'PI', 'Trial_Type': 'NW', 
                    'Cue_Valence': 1, 'Optimal_Action': 0}] * n_nw)
    # GAL: Avoid Cue (-1), Optimal Action = Go (1)
    trials.extend([{'Block': block_name, 'Conflict': 'PI', 'Trial_Type': 'GAL', 
                    'Cue_Valence': -1, 'Optimal_Action': 1}] * n_gal)
    
    # 4. Shuffle the trials so they appear in a random order within the block
    df = pd.DataFrame(trials)
    df = df.sample(frac=1).reset_index(drop=True)
    return df

# ==========================================
# Generate the Full Experiment
# ==========================================
np.random.seed(42) # For reproducibility

# Set how many trials you want per block (e.g., 40 trials * 4 blocks = 160 total trials)
TRIALS_PER_BLOCK = 500

# Generate the 4 specific blocks
b1 = create_block('B1_MC', TRIALS_PER_BLOCK, pc_ratio=0.50)
b2 = create_block('B2_HC1', TRIALS_PER_BLOCK, pc_ratio=0.30)
b3 = create_block('B3_HC2', TRIALS_PER_BLOCK, pc_ratio=0.30)
b4 = create_block('B4_LC', TRIALS_PER_BLOCK, pc_ratio=0.70)

# Combine them sequentially
experiment_df = pd.concat([b1, b2, b3, b4], ignore_index=True)

# Add Trial Numbers and Placeholders for the Agent
experiment_df.insert(0, 'Trial', experiment_df.index + 1)
experiment_df['Agent_Choice'] = np.nan
experiment_df['Agent_Reward'] = np.nan

# ==========================================
# Verification
# ==========================================
print("--- Experiment Structure Verification ---")
# Count the PC vs PI trials in each block to prove the math is correct
summary = experiment_df.groupby(['Block', 'Conflict']).size().unstack(fill_value=0)
summary['% PC'] = (summary['PC'] / (summary['PC'] + summary['PI'])) * 100
print(summary)

print("\n--- First 10 Trials of Block 1 ---")
print(experiment_df.head(50))

# Save the pure environment
# experiment_df.to_csv('pure_task_environment.csv', index=False)

--- Experiment Structure Verification ---
Conflict   PC   PI  % PC
Block                   
B1_MC     250  250  50.0
B2_HC1    150  350  30.0
B3_HC2    150  350  30.0
B4_LC     350  150  70.0

--- First 10 Trials of Block 1 ---
    Trial  Block Conflict Trial_Type  Cue_Valence  Optimal_Action  \
0       1  B1_MC       PI         NW            1               0   
1       2  B1_MC       PC         GW            1               1   
2       3  B1_MC       PI         NW            1               0   
3       4  B1_MC       PC        NAL           -1               0   
4       5  B1_MC       PC         GW            1               1   
5       6  B1_MC       PI        GAL           -1               1   
6       7  B1_MC       PI        GAL           -1               1   
7       8  B1_MC       PC         GW            1               1   
8       9  B1_MC       PC         GW            1               1   
9      10  B1_MC       PI        GAL           -1               1   
10     11  B1

In [21]:
from scipy.special import expit as inv_logit

params = {
    'xi': 0.1, #noise rate, randomness
    'ep': 0.2, # learning rate
    'b':  0.7, # go bias
    'pi': 3.0, # pavlovian bias
    'rho': 2.0 # reward sensitivity
}

def decide(params, qv_g, qv_ng, sv):
    # sv = stimulus value
    # ep = learning rate
    # rho = rew sensitivity
    # outcome = the rew received this trial (+1,0,1)

    b=params['b']
    pi=params['pi']
    wv_g  = qv_g + b + pi * sv
    wv_ng = qv_ng
    pGo = inv_logit(wv_g - wv_ng)
    pGo = pGo*(1-params['xi'])
    pGo = pGo+(params['xi']/2)
    return pGo
    
float(decide(params, qv_g=0, qv_ng=0.8, sv=0.8))
# setting bias(b) = 3 and sv(cue predicts rew) = 0.8
# we got a strong go bias, ie cue for rew+pav bias is high, means 
# more tendency to go 
# high value of qv_ng implies nogo is correct. 
# the agent will keep making error on conflict trials.

def update (params, sv, qv_g, qv_ng, outcome, pressed):
    sv = sv + params['ep']*(params['rho']*outcome - sv)
    #Q-value update lines
    if (pressed):
        qv_g = qv_g + params['ep']*(params['rho']*outcome-qv_g)
    else:
        qv_ng = qv_ng + params['ep']*(params['rho']*outcome-qv_ng)
    return qv_g, qv_ng, sv



In [27]:
def run_agent():
    # dictionary per cue
    cues=['GW','NAL','NW','GAL']
    qv_g  = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    qv_ng = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    sv    = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    for idx, row in experiment_df.iterrows():
        cue = row['Trial_Type']
        pGo=decide(qv_g[cue],qv_ng[cue],sv[cue])
        action=np.random.binomial(1,pGo)
        experiment_df.at[idx, 'Agent_Choice'] = action
    